In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('bridges.csv')
print(f"Rows: {df.shape[0]}, Columns: {df.shape[1]}")
df.head()


Rows: 108, Columns: 13


,Id,river,location,erected,purpose,length,lanes,clear-g,t-or-d,material,span,rel-l,type
0,E1,M,3.0,1818,HIGHWAY,NaN,2.0,N,THROUGH,WOOD,SHORT,S,WOOD
1,E2,A,25.0,1819,HIGHWAY,1037.0,2.0,N,THROUGH,WOOD,SHORT,S,WOOD
2,E3,A,39.0,1829,AQUEDUCT,NaN,1.0,N,THROUGH,WOOD,NaN,S,WOOD
3,E5,A,29.0,1837,HIGHWAY,1000.0,2.0,N,THROUGH,WOOD,SHORT,S,WOOD
4,E6,M,23.0,1838,HIGHWAY,NaN,2.0,N,THROUGH,WOOD,NaN,S,WOOD


## 1. Look at the data before assuming anything

`length` is the only numeric attribute in this dataset (feet). Everything else (`river`, `purpose`,
`material`, `span`, `type`, etc.) is categorical or an identifier. Before stating a plausibility rule,
it's worth knowing (a) how much data is missing and (b) the overall shape of the distribution — a
rule should be informed by what "normal" looks like for *these* bridges, not an arbitrary number.


In [2]:
missing = df['length'].isna().sum()
print(f"Missing 'length' values: {missing} of {len(df)} rows ({missing/len(df):.1%})")
print()
df['length'].describe()


Missing 'length' values: 27 of 108 rows (25.0%)



count      81.000000
mean     1567.469136
std       747.491523
min       804.000000
25%      1000.000000
50%      1300.000000
75%      2000.000000
max      4558.000000
Name: length, dtype: float64

Two things worth noting:

- **27 rows (25%) have no recorded length at all.** These are *missing* values, not outliers — there is
  nothing implausible about a blank cell, so they are not part of the outlier screen below. They are a
  separate data-quality issue (handled at the end).
- Among the 81 rows that do have a length, the values range from 804 ft to 4,558 ft, with a median around
  1,300 ft. No length is zero or negative, so there's no "obviously impossible" value the way Lab 2 had —
  this is exactly the subtler situation the assignment describes.


## 2. Stated assumption (before screening)

**Context:** these are 19th- and early-20th-century highway and railroad bridges in the Pittsburgh area,
crossing the Allegheny, Monongahela, and Ohio rivers. `length` is the total structure length in feet.

**Plausibility rule, stated before I look at what it flags:**

> A bridge `length` is plausible if it falls between **200 ft and 3,000 ft**.
>
> - **Lower bound (200 ft):** a structure recorded in this dataset as a "bridge" spanning a river
>   should be at least long enough to actually cross open water — a few dozen feet would suggest a
>   culvert or a recording error, not a river bridge.
> - **Upper bound (3,000 ft):** the vast majority of 19th-century wood/iron/steel truss and suspension
>   bridges in this region were in the few-hundred-to-low-thousands-of-feet range. 3,000 ft is a generous
>   ceiling — well above the 75th percentile (2,000 ft) — chosen so that only *unusually* long structures
>   get flagged, not just the long tail of an already-long-tailed distribution.

Rows outside `[200, 3000]` are flagged below. I am committing to this rule **before** running the
screen, per the assignment's instructions.


In [3]:
LOWER_BOUND = 200
UPPER_BOUND = 3000

has_length = df['length'].notna()
implausible = has_length & ((df['length'] < LOWER_BOUND) | (df['length'] > UPPER_BOUND))

print(f"Rows screened (non-missing length): {has_length.sum()}")
print(f"Rows flagged as implausible:         {implausible.sum()}")
print()
flagged = df.loc[implausible, ['Id', 'river', 'erected', 'purpose', 'length', 'material', 'span', 'type']] \
            .sort_values('length', ascending=False)
flagged


Rows screened (non-missing length): 81
Rows flagged as implausible:         3



,Id,river,erected,purpose,length,material,span,type
31,E34,O,1888,RR,4558.0,STEEL,LONG,SIMPLE-T
44,E46,A,1897,RR,4000.0,STEEL,LONG,SIMPLE-T
104,E91,O,1975,HIGHWAY,3756.0,STEEL,LONG,ARCH


## 3. What got flagged

Three rows exceed the 3,000 ft upper bound: **E34** (4,558 ft), **E46** (4,000 ft), and **E91** (3,756 ft).
No rows fall below the 200 ft lower bound — every recorded length is at least 804 ft, so the lower bound
did not end up screening anything out for this particular dataset (it's still correct to state it, since a
rule chosen *after* seeing that no row is short would be reasoning backwards).


In [4]:
# A closer look at *why* the three flagged rows are so long
flagged[['Id', 'river', 'erected', 'purpose', 'material', 'span', 'type', 'length']]


,Id,river,erected,purpose,material,span,type,length
31,E34,O,1888,RR,STEEL,LONG,SIMPLE-T,4558.0
44,E46,A,1897,RR,STEEL,LONG,SIMPLE-T,4000.0
104,E91,O,1975,HIGHWAY,STEEL,LONG,ARCH,3756.0


## 4. Handling the flagged rows, and why

I am **not deleting or capping** these three rows. Here's the reasoning:

- All three (`E34`, `E46`, `E91`) have `span == LONG` and `material == STEEL`. That combination is exactly
  what you'd expect to *legitimately* produce a length far outside the typical wood/iron truss range: a
  long-span steel bridge (e.g. a railroad bridge crossing the Ohio River, or the 1975 highway bridge
  `E91`) is structurally capable of — and expected to — run several thousand feet, especially for a
  multi-span railroad crossing.
- In other words, my *statistical* rule (3,000 ft ceiling) correctly flagged these as unusual relative to
  the rest of the dataset, but the *domain* evidence in the other columns (`span`, `material`, `purpose`)
  supports treating them as real, valid measurements rather than data-entry errors — there's no column
  where a "4,558" could plausibly be a typo for something in-range (e.g. a decimal-point slip would need
  to be off by 10x, which isn't a typical typo pattern).
- **Decision:** keep all three rows unchanged in the analysis dataset, but retain the boolean
  `length_flag` column below so any downstream analysis (e.g. an average-length-by-material calculation)
  can choose to exclude or separately weight them if their scale would distort a mean or a chart.

Separately, the **27 missing `length` values** are handled differently, since they are a missing-data
problem rather than an outlier problem: I leave them as `NaN` (not imputed to 0 or to the mean) so that
any later aggregation on `length` correctly excludes them via `.dropna()`/`.mean()` rather than being
silently biased by a fabricated fill value.


In [5]:
df['length_flag'] = implausible
print("Value counts for the new 'length_flag' column:")
print(df['length_flag'].value_counts())
print()
kept = (~df['length_flag']).sum()
print(f"Rows kept as-is (not flagged, including the 27 with missing length): {kept} / {len(df)}")


Value counts for the new 'length_flag' column:
length_flag
False    105
True       3
Name: count, dtype: int64

Rows kept as-is (not flagged, including the 27 with missing length): 105 / 108


## 5. Summary

- **Assumption (stated first):** a plausible bridge length is between 200 ft and 3,000 ft, based on the
  physical need to span a river and the typical scale of 19th-century truss/suspension bridges in this
  region.
- **Screen:** applied to the 81 non-missing `length` values; flagged 3 of them (E34, E46, E91), all > 3,000 ft.
- **Handling:** kept all three flagged rows — the domain columns (`span=LONG`, `material=STEEL`) support
  them as genuine long-span steel bridges rather than data-entry errors — and added a `length_flag` column
  so they can be excluded from length-sensitive summary statistics later if needed. The 27 missing lengths
  are left as `NaN`, since they are a separate missing-data issue, not an outlier.

---
**GitHub repo:** https://github.com/roee0910/cs82a-portfolio.git
